In [ ]:
%pip install -r requirements.txt

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

from langchain_huggingface import HuggingFaceEmbeddings

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_chroma import Chroma

C:\Users\navee\AppData\Local\Temp\ipykernel_3528\638219830.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [2]:
loader = PyPDFDirectoryLoader("./PDFS",)

In [ ]:
pdf = loader.load()


[Document(metadata={'producer': 'Internet Archive PDF 1.4.24; including mupdf and pymupdf/skimage', 'creator': 'Internet Archive (Scribe Version 6.4-initial-3-g9590e5ec)', 'creationdate': '2023-11-29T07:40:17+00:00', 'author': 'Swift, Jonathan, 1667-1745', 'ebx_publisher': 'Maidenhead, [Eng.] : Purnell', 'keywords': 'https://archive.org/details/gulliverstravels0000swif_x6o2', 'moddate': '2023-11-29T07:40:17+00:00', 'subject': 'Voyages and travels -- Fiction; Fantasy', 'title': "Gulliver's travels", 'source': "PDFS\\Gulliver's travels, part one, a voyage to Lilliput ; [and] -- Jonathan Swift; ill_ by Derick Bowen [i_e_ Bown] -- A Purnell de luxe classic, -- 9780361031585 -- 7ab3df9948c3b6cd16a4e9423adba343 -- Anna’s Archive.pdf", 'total_pages': 182, 'page': 0, 'page_label': ''}, page_content='GULLIVER’S | \nTRAVEL, \nee'),
 Document(metadata={'producer': 'Internet Archive PDF 1.4.24; including mupdf and pymupdf/skimage', 'creator': 'Internet Archive (Scribe Version 6.4-initial-3-g9590e5

In [4]:
text_Splitters = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

In [5]:
documents = text_Splitters.split_documents(pdf)
len(documents)

401

In [ ]:
filtered_docs = [doc for doc in documents if len(doc.page_content.strip("\\s+")) > 80]

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\navee\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11136.66it/s]


In [ ]:

if vector_store._collection.count() == 0:
    vector_store = Chroma(collection_name="Gulliver_Travels",embedding_function=embedding_model,persist_directory="./vector_store",)
    vector_store.add_documents(documents=filtered_docs)
else:
    print(f"vector store already contains documents of {vector_store._collection.count()} skipping re-indexing.")

vector store already contains documents of 802 skipping re-indexing.


In [13]:
Query = "Who is Gulliver?"

In [18]:
docs = vector_store.similarity_search(Query,k=5)
for i, doc in enumerate(docs):
    print(f"{i} -> {doc.page_content}")

0 -> GULLIVER’S TRAVELS
1 -> Gulliver’s Travels
2 -> GULLIVER’S TRAVELS
3 -> Gulliver’s Travels
4 -> GULLIVER’S | 
TRAVEL, 
ee
